In [ ]:
import os
import re
import glob
import time
import matplotlib.pyplot as plt
import dask.array as da
import numpy as np
import json
import tifffile

import zarr
from ngff_zarr import (
    detect_cli_io_backend,
    cli_input_to_ngff_image,
    to_multiscales,
    to_ngff_zarr,
    Methods,
    config,
    NgffImage,
)

import zarr.storage

zarr.storage.BaseStore = object

class DirectoryStore:
    def __init__(self, path, dimension_separator="/", **kwargs):
        self._store = zarr.storage.LocalStore(path, **kwargs)
    def __getattr__(self, name):
        return getattr(self._store, name)

zarr.storage.DirectoryStore = DirectoryStore
zarr.DirectoryStore = DirectoryStore

print(zarr.storage.DirectoryStore)
print(zarr.storage.DirectoryStore("../zarr-ngff-cache", dimension_separator="/"))

In [ ]:
recon_dirs = ["/pscratch/sd/t/tachavez/SYNAPS/data/rec20260221_103725_petiole15"]
seg_dir = "/global/cfs/cdirs/als/synaps/beamtime_022126/combined/extract_regions/Dehydrated_Xylem_vessels"

# Find subdirectories containing TIFF files
seg_dirs = []
for root, dirs, files in os.walk(seg_dir):
    tiff_files = glob.glob(os.path.join(root, "*.tif")) + glob.glob(
        os.path.join(root, "*.tiff")
    )
    if tiff_files:
        if "masks" in root:
            seg_dirs.append(root)

In [ ]:
recon_dirs

In [ ]:
seg_dirs

In [ ]:
# Check on the datatype of the tiffs in a folder
def inspect_tiff_dtype(image_dir, sample_count=3):
    file_names = [
        f
        for f in os.listdir(image_dir)
        if f.lower().endswith((".tif", ".tiff")) and not f.startswith(".")
    ]
    file_names.sort()
    if not file_names:
        print(f"No TIFF files found in {image_dir}")
        return
    sample_files = file_names[:sample_count]
    dtypes = []
    for fname in sample_files:
        fpath = os.path.join(image_dir, fname)
        with tifffile.TiffFile(fpath) as tf:
            arr = tf.asarray()
            dtypes.append(arr.dtype)
    unique_dtypes = sorted(set(dtypes), key=lambda x: str(x))
    print(f"{os.path.basename(image_dir)} -> {unique_dtypes}")


# Example: inspect a few reconstruction folders
for image_dir in sorted(set(recon_dirs))[:5]:
    inspect_tiff_dtype(image_dir)

for image_dir in sorted(set(seg_dirs))[:5]:
    inspect_tiff_dtype(image_dir)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


def find_closest_to_white(rgb_points):
    """
    Given a flat list of RGB points [pos, r, g, b, ...],
    find the point closest to white (1.0, 1.0, 1.0).

    Returns:
        tuple: (pos, r, g, b) closest to white
    """
    # Convert to list of tuples
    tuples = [tuple(rgb_points[i : i + 4]) for i in range(0, len(rgb_points), 4)]

    # Define white in RGB
    white = np.array([1.0, 1.0, 1.0])

    # Compute distances to white
    distances = [np.linalg.norm(np.array([r, g, b]) - white) for _, r, g, b in tuples]

    # Find index of closest point
    closest_idx = np.argmin(distances)
    return tuples[closest_idx]


def load_custom_colormap_from_json_list(json_path, colormap_name):
    """
    Load a custom matplotlib colormap from a JSON file containing a list of colormap definitions.

    Parameters:
        json_path (str): Path to the JSON file.
        colormap_name (str): Name of the colormap to load.

    Returns:
        LinearSegmentedColormap: A matplotlib-compatible colormap.
    """
    # Load JSON list
    with open(json_path, "r") as f:
        colormap_list = json.load(f)

    # Search for the colormap by name
    selected_cmap = next(
        (cmap for cmap in colormap_list if cmap.get("Name") == colormap_name), None
    )

    if not selected_cmap:
        raise ValueError(f"Colormap '{colormap_name}' not found in JSON file.")

    rgb_points = selected_cmap["RGBPoints"]
    print(f"Closest to white {find_closest_to_white(rgb_points)}")

    # Convert flat list to (position, r, g, b) tuples
    tuples = [tuple(rgb_points[i : i + 4]) for i in range(0, len(rgb_points), 4)]

    # Build color dict
    cdict = {"red": [], "green": [], "blue": []}
    for pos, r, g, b in tuples:
        cdict["red"].append((pos, r, r))
        cdict["green"].append((pos, g, g))
        cdict["blue"].append((pos, b, b))

    # Create and return colormap
    return LinearSegmentedColormap(colormap_name, segmentdata=cdict)


# cmap=load_custom_colormap_from_json_list("ColorMaps.json", "Asymmetrical Earth Tones (6_21b)")
cmap = load_custom_colormap_from_json_list(
    "ColorMaps.json", "Green-Blue Asymmetric Divergent (62Blbc)"
)

In [ ]:
def get_min_max_tiffstack(image_dir, percentile_bins=2048):
    start_time = time.time()
    file_names = os.listdir(image_dir)
    file_paths = [
        os.path.join(image_dir, file_name)
        for file_name in file_names
        if not file_name.startswith(".")
    ]
    file_paths.sort()
    backend = detect_cli_io_backend(file_paths)
    image = cli_input_to_ngff_image(backend, file_paths)
    min_value = image.data.min().compute()
    max_value = image.data.max().compute()

    if min_value == max_value:
        p01 = p99 = min_value
    else:
        hist_p, bin_edges_p = da.histogram(
            image.data, bins=percentile_bins, range=[min_value, max_value]
        )
        hist_p = hist_p.compute()
        cdf = np.cumsum(hist_p)
        if cdf[-1] == 0:
            p01 = p99 = min_value
        else:
            cdf = cdf / cdf[-1]
            p01 = np.interp(0.01, cdf, bin_edges_p[1:])
            p99 = np.interp(0.99, cdf, bin_edges_p[1:])

    histogram = da.histogram(image.data, bins=20, range=[min_value, max_value])
    end_time = time.time()
    print(
        f"Min: {min_value}, Max: {max_value}, P01: {p01}, P99: {p99}, calculated in {end_time-start_time:.2f} seconds"
    )
    return min_value, max_value, p01, p99, histogram

In [ ]:
def compute_out_of_range_percentage(dask_array, min_value, max_value):
    total_count = dask_array.size
    below_min = (dask_array < min_value).sum()
    above_max = (dask_array > max_value).sum()
    out_of_range = below_min + above_max

    # Compute the result
    out_of_range_count = out_of_range.compute()
    percentage = (out_of_range_count / total_count) * 100

    return percentage

In [ ]:
def convert_to_uint8(slice_data, min_val, max_val):
    clipped = np.clip(slice_data, min_val, max_val)
    scaled = (clipped - min_val) / (max_val - min_val)
    return (scaled * 255).astype(np.uint8)


def extract_and_plot_orthogonal_slices_dask(
    image,
    name,
    min_value=-10,
    max_value=10,
    colormap="gist_earth",
    output_dir="./plots",
):
    """
    Extracts center XY, XZ, and YZ slices from 3D Dask arrays (NgffImages)
    and saves them in a single subplot figure.

    Parameters:
        image (NgffImage): contains a 3D Dask arrays.
        output_dir (str): Directory to save output image.
    """
    os.makedirs(output_dir, exist_ok=True)

    data = image.data  # Dask array with shape (Z, Y, X)

    try:
        # Compute center indices
        center_z = data.shape[0] // 2
        center_y = data.shape[1] // 2
        center_x = data.shape[2] // 2

        # Extract and compute the slices
        axial = data[center_z, :, :].compute()  # XY slice
        axialtop = data[center_z - center_z // 2, :, :].compute()  # XY slice top
        axialbottom = data[center_z + center_z // 2, :, :].compute()  # XY slice bottom
        coronal = data[:, center_y, :].compute()  # XZ slice
        sagittal = data[:, :, center_x].compute()  # YZ slice
    except Exception as e:
        print(f"❌ Failed to extract slices from {name}: {e}")
        return

    # Plot all three slices
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    im = axes[0][0].imshow(axial, cmap=colormap, vmin=min_value, vmax=max_value)
    axes[0][0].set_title("XY-center slice (original tiffs)")
    axes[0][1].imshow(axialtop, cmap=colormap, vmin=min_value, vmax=max_value)
    axes[0][1].set_title("XY-quarter top slice (original tiffs)")
    im2 = axes[0][2].imshow(axialbottom, cmap=colormap, vmin=min_value, vmax=max_value)
    axes[0][2].set_title("XY-quarter bottom slice (original tiffs)")

    axes[1][0].imshow(coronal, cmap=colormap, vmin=min_value, vmax=max_value)
    axes[1][0].set_title(" slice")
    axes[1][1].imshow(sagittal, cmap=colormap, vmin=min_value, vmax=max_value)
    axes[1][1].set_title("YZ-center slice")

    cbar = fig.colorbar(
        im, ax=axes[0][0], orientation="vertical", fraction=0.025, pad=0.04
    )
    cbar.set_label("Intensity")

    # Compute Dask histogram over entire volume
    try:
        hist, bin_edges = da.histogram(data, bins=100, range=(-2, 2))
        hist_vals = hist.compute()
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        axes[1][2].bar(
            bin_centers, hist_vals, width=(bin_edges[1] - bin_edges[0]), color="black"
        )
        axes[1][2].set_yscale("log")
        axes[1][2].set_title("Intensity Histogram")
        axes[1][2].set_xlabel("Intensity")
        axes[1][2].set_ylabel("Frequency")
    except Exception as e:
        print(f"⚠️ Histogram failed: {e}")
        axes[1][2].set_title("Histogram Error")

    # Save plot
    out_path = os.path.join(output_dir, f"{name}_center_slices.png")
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0.1)
    plt.close()

    # Convert to uint8 for visual comparison
    axial_u8 = convert_to_uint8(axial, min_value, max_value)
    coronal_u8 = convert_to_uint8(coronal, min_value, max_value)
    sagittal_u8 = convert_to_uint8(sagittal, min_value, max_value)

    # Convert uint8 back to float32 normalized in [0, 1]
    axial_u8_rescaled = axial_u8.astype(np.float32) / 255.0
    coronal_u8_rescaled = coronal_u8.astype(np.float32) / 255.0
    sagittal_u8_rescaled = sagittal_u8.astype(np.float32) / 255.0

    # Original float32 values normalized to [0, 1]
    axial_float_norm = np.clip((axial - min_value) / (max_value - min_value), 0, 1)
    coronal_float_norm = np.clip((coronal - min_value) / (max_value - min_value), 0, 1)
    sagittal_float_norm = np.clip(
        (sagittal - min_value) / (max_value - min_value), 0, 1
    )

    # Compute differences (in normalized scale)
    axial_diff = axial_float_norm - axial_u8_rescaled
    coronal_diff = coronal_float_norm - coronal_u8_rescaled
    sagittal_diff = sagittal_float_norm - sagittal_u8_rescaled

    vmax = max(
        np.abs(axial_diff).max(),
        np.abs(coronal_diff).max(),
        np.abs(sagittal_diff).max(),
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 5))

    vmax = np.abs(axial_diff).max()
    im = axes[0][0].imshow(axial_diff, cmap="bwr", vmin=-vmax, vmax=vmax)
    axes[0][0].set_title("XY-center Difference (float32 - uint8)")
    cbar = fig.colorbar(
        im, ax=axes[0][0], orientation="vertical", fraction=0.025, pad=0.04
    )

    vmax = np.abs(coronal_diff).max()
    im1 = axes[0][1].imshow(coronal_diff, cmap="bwr", vmin=-vmax, vmax=vmax)
    axes[0][1].set_title("XZ-center Difference (float32 - uint8)")
    cbar = fig.colorbar(
        im1, ax=axes[0][1], orientation="vertical", fraction=0.025, pad=0.04
    )

    vmax = np.abs(sagittal_diff).max()
    im2 = axes[0][2].imshow(sagittal_diff, cmap="bwr", vmin=-vmax, vmax=vmax)
    axes[0][2].set_title("YZ-center Difference (float32 - uint8)")
    cbar = fig.colorbar(
        im2, ax=axes[0][2], orientation="vertical", fraction=0.025, pad=0.04
    )

    im = axes[1][0].imshow(axial_u8, cmap=colormap)
    axes[1][0].set_title("XY-center (uint8)")
    cbar = fig.colorbar(
        im, ax=axes[1][0], orientation="vertical", fraction=0.025, pad=0.04
    )
    im1 = axes[1][1].imshow(coronal_u8, cmap=colormap)
    axes[1][1].set_title("XZ-center (uint8)")
    cbar = fig.colorbar(
        im1, ax=axes[1][1], orientation="vertical", fraction=0.025, pad=0.04
    )
    im2 = axes[1][2].imshow(sagittal_u8, cmap=colormap)
    axes[1][2].set_title("YZ-center (uint8)")
    cbar = fig.colorbar(
        im2, ax=axes[1][2], orientation="vertical", fraction=0.025, pad=0.04
    )

    plt.tight_layout()
    # Save plot
    out_path = os.path.join(output_dir, f"{name}_center_slices_diff.png")
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0.1)
    plt.close()

    print(f" Saved: {out_path}")

In [ ]:
def plotslices(image_dir, min_value=-2, max_value=2, colormap="twilight"):
    file_names = os.listdir(image_dir)
    file_paths = [
        os.path.join(image_dir, file_name)
        for file_name in file_names
        if not file_name.startswith(".")
    ]
    file_paths.sort()
    backend = detect_cli_io_backend(file_paths)
    image = cli_input_to_ngff_image(backend, file_paths)
    extract_and_plot_orthogonal_slices_dask(
        image, os.path.basename(image_dir), min_value, max_value, colormap=colormap
    )

In [ ]:
for image_dir in sorted(set(recon_dirs)):
    start_time = time.time()
    # Get min/max values for the current image directory
    min_val, max_val, p01, p99, histogram = get_min_max_tiffstack(image_dir)
    plotslices(image_dir, min_value=p01, max_value=p99, colormap=cmap)
    end_time = time.time()
    print(f"Took {end_time-start_time:.2f} seconds")

In [ ]:
def normalize_to_uint8(data, selected_min, selected_max):
    return data.map_blocks(
        lambda block: (
            np.clip((block - selected_min) / (selected_max - selected_min), 0, 1) * 255
        ).astype(np.uint8),
        dtype=np.uint8,
    )

In [ ]:
def create_zarr(image_dir, selected_min, selected_max, dtype="uint8"):
    file_names = os.listdir(image_dir)
    file_paths = [
        os.path.join(image_dir, file_name)
        for file_name in file_names
        if not file_name.startswith(".")
    ]
    file_paths.sort()
    backend = detect_cli_io_backend(file_paths)
    image = cli_input_to_ngff_image(backend, file_paths)
    config.cache_store = zarr.storage.DirectoryStore(
        "../zarr-ngff-cache", dimension_separator="/"
    )

    data_dtype = image.data.dtype
    if data_dtype == np.uint8:
        image_uint8 = image
        multiscale_method = Methods.DASK_IMAGE_NEAREST
    else:
        image_uint8 = NgffImage(
            data=normalize_to_uint8(image.data, selected_min, selected_max),
            dims=image.dims,
            scale=image.scale,
            translation=image.translation,
            name=image.name,
            axes_units=image.axes_units,
            computed_callbacks=image.computed_callbacks,
        )
        multiscale_method = Methods.DASK_IMAGE_GAUSSIAN

    zarrs_dir = os.path.abspath(os.path.join(image_dir, "..", "zarrs"))
    os.makedirs(zarrs_dir, exist_ok=True)
    out_name = os.path.basename(image_dir)
    if dtype == "uint8":
        out_name = f"{out_name}_{dtype}"
    out_zarr_directory = os.path.join(zarrs_dir, f"{out_name}.zarr")
    start_time = time.time()
    multiscales = to_multiscales(
        image_uint8 if dtype == "uint8" else image, method=multiscale_method
    )
    to_ngff_zarr(out_zarr_directory, multiscales)
    end_time = time.time()
    print(f"NGFF Zarr for {os.path.basename(image_dir)} with dtype {dtype} created")
    float32_time = end_time - start_time
    print(f"Execution time: {float32_time:.2f} seconds")
    print("Output directory: " + out_zarr_directory)

In [ ]:
recon_dirs

In [ ]:
for image_dir in sorted(set(recon_dirs)):
    start_time = time.time()
    create_zarr(image_dir, selected_min=p01, selected_max=p99, dtype="uint8")
    end_time = time.time()
    print(f"Execution time total: {end_time-start_time:.2f} seconds")

In [ ]:
seg_dirs

In [ ]:
for image_dir in sorted(set(seg_dirs)):
    start_time = time.time()
    create_zarr(image_dir, selected_min=0, selected_max=255, dtype="uint8")
    end_time = time.time()
    print(f"Execution time total: {end_time-start_time:.2f} seconds")

In [ ]:
zarr_directory = "/global/cfs/cdirs/als/synaps/combined/extract_regions/Dehydrated_Xylem_vessels/zarrs/masks_uint8.zarr"

multiscales = zarr.open(zarr_directory, mode='r')

In [ ]:
import matplotlib.pyplot as plt
import zarr
import numpy as np

arr = zarr.open(zarr_directory, mode='r')

scale0_data = arr['scale0']['image']
scale2_data = arr['scale4']['image']

mid0 = scale0_data.shape[0] // 2
mid2 = scale2_data.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(scale0_data[mid0], cmap='gray')
axes[0].set_title(f'Scale 0 (full res) - shape: {scale0_data.shape}')

axes[1].imshow(scale2_data[mid2], cmap='gray')
axes[1].set_title(f'Scale 2 (downsampled) - shape: {scale2_data.shape}')

plt.tight_layout()
plt.show()